# Benchmarking dependency measures for marker-gene identification in multi-modal single-cell data

**PbenchmarkC CITE-seq (Hao et al. 2021) — a 2×2 decomposition of dependency measures.**
**Granularity: L2 (~30 cell types).**

|            | Marginal                | Conditional                       |
|------------|-------------------------|-----------------------------------|
| **Linear**    | Spearman correlation    | Partial correlation (shrinkage)   |
| **Nonlinear** | Mutual information (KSG) | Integrated Gradients on an MLP     |

We ask whether *nonlinearity* and *multivariate context* change which genes are
identified as cell-type markers, and which axis matters more. Every method sees
**RNA only**; the surface-protein (ADT) modality is used solely to define a
cross-modal, protein-derived ground-truth driver set $D_c$ per cell type. Recovery
of $D_c$ is scored with the parameter-free $\mathrm{AUC_{rel}}$ metric.

All heavy logic lives in the importable `src/` package.
This notebook only shows the good parts and presents the narrative.

See `README.md` for the full list of design decisions.


In [4]:
%reload_ext autoreload
%autoreload 2

# import torch
# from types import SimpleNamespace
import numpy as np
import pandas as pd
from src import (
        config,
        data,
        preprocessing,
        ground_truth,
        protein_gene_map,
        # scorers,
        benchmark,
        stats,
        metric,
        plotting
)

# Turn off logging
config.set_log_level_critical()
plotting.set_style()
plotting.set_fig_level("l2")
LEVEL = "celltype.l2"
TAG = ""
rng = np.random.default_rng(config.SEED)

## 1. The Data

We use data from made available by Hao 2021 3 (CITE-seq data, GEO `GSE164378`: whole-transcriptome RNA + 228
surface proteins + donor / `celltype.l1/l2/l3` labels).

It was downloaded and subsampled to ~25k cells using **sqrt-proportional stratified subsampling** and saved as a MuData dataset (.h5mu – annotated multimodal data). This was done once, reducing the data size from ~1.4G to ~500M and locally saved.

## Load the data

In [48]:
# Load the multi modal dataset
dataset = data.create_or_load_dataset()
rna, adt = dataset["rna"], dataset["adt"]

### Some descriptive statistics

In [33]:
print(f"Cell count: {rna.n_obs:,}")
print(f"RNA: {rna.n_vars:,} genes")
print(f"ADT: {adt.n_vars} proteins")

Cell count: 24,735
RNA: 33,538 genes
ADT: 228 proteins


#### RNA Data

The matrix:
* rows = observations (cells) (~25k)
* cols = variables (genes) (~33k)

In [34]:
rna.X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 53203323 stored elements and shape (24735, 33538)>

**Observation (row) annotation data**
This is the annotation data for the observations/rows — different metadata about cells.

In [49]:
rna.obs

,nCount_ADT,nFeature_ADT,nCount_RNA,nFeature_RNA,orig.ident,lane,donor,time,celltype.l1,celltype.l2,celltype.l3,Phase,Batch
L1_AAACCCACACGTACTA,3567,202,4786,1890,SeuratProject,L1,P3,7,NK,NK,NK_2,G1,Batch1
L1_AAACCCACATGGATCT,8210,222,3589,1122,SeuratProject,L1,P4,2,B,B intermediate,B intermediate lambda,G1,Batch1
L1_AAACCCATCTTAAGGC,5382,215,4016,1451,SeuratProject,L1,P2,2,CD4 T,CD4 CTL,CD4 CTL,S,Batch1
L1_AAACGAAAGATAACAC,5155,216,3393,1092,SeuratProject,L1,P2,2,B,B naive,B naive kappa,S,Batch1
L1_AAACGAACAATGAGCG,5465,211,7379,2315,SeuratProject,L1,P1,7,CD4 T,CD4 TCM,CD4 TCM_2,G1,Batch1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
E2L8_TTTGGAGAGGAAGAAC,1919,193,4140,1669,SeuratProject,E2L8,P5,0,NK,NK,NK_2,S,Batch2
E2L8_TTTGGTTAGGCCTGCT,3392,200,7966,2028,SeuratProject,E2L8,P6,0,CD4 T,CD4 Naive,CD4 Naive,G1,Batch2
E2L8_TTTGGTTTCAATCCAG,3427,190,8034,1982,SeuratProject,E2L8,P6,7,CD8 T,CD8 Naive,CD8 Naive,S,Batch2
E2L8_TTTGTTGCAGCGTGAA,3426,205,7225,2131,SeuratProject,E2L8,P8,2,CD4 T,CD4 Naive,CD4 Naive,G1,Batch2


**Variable (column) annotation data**
This is the annotation data for the columns — the gene names.

In [52]:
rna.var_names
adt.var_names

# np.unique([1,1,3])

Index(['adt:CD39', 'adt:Rat-IgG1-1', 'adt:CD107a', 'adt:CD62P', 'adt:TCR-2',
       'adt:CD30', 'adt:CD31', 'adt:CD34', 'adt:CD35', 'adt:CD36',
       ...
       'adt:CD169', 'adt:CD28', 'adt:CD161', 'adt:CD163', 'adt:CD138-1',
       'adt:CD164', 'adt:CD138-2', 'adt:CD144', 'adt:CD202b', 'adt:CD11c'],
      dtype='object', length=228)

## 2. Transform and process the data Preprocessing and the shared feature matrix

Standard Scanpy/muon workflow: QC, `log1p` for RNA, CLR for ADT, HVG selection.
The four methods then consume the **identical** matrix `X` — the
rank-transformed, z-scored expression of the shared gene universe
(**HVGs ∪ protein-encoding marker genes**). The per-gene average-rank transform
also collapses dropout zeros to a shared rank, mitigating the zero-inflation
confound for MI.

In [1]:
preprocessing.compute_qc(rna)
preprocessing.normalize_rna(rna)
preprocessing.normalize_adt(adt)
hvg = preprocessing.select_hvg(rna)
marker_genes = protein_gene_map.all_candidate_genes(adt.var_names) & set(rna.var_names)
universe = sorted(set(hvg) | marker_genes)
X, genes = preprocessing.build_shared_matrix(rna, universe)
print(f"HVGs {len(hvg)} | marker genes {len(marker_genes)} | shared universe {len(genes)}")
print("shared matrix:", X.shape, X.dtype)

NameError: name 'preprocessing' is not defined

In [ ]:
qc = rna.obs[["n_genes_by_counts","total_counts","pct_counts_mito"]].rename(
    columns={"n_genes_by_counts":"genes / cell","total_counts":"UMI / cell",
             "pct_counts_mito":"% mitochondrial"})
fig = plotting.qc_violins(qc, list(qc.columns)); plotting.save(fig, f"fig_qc{TAG}"); fig

## 3. Exploration: UMAP, donor batch effect, cell-type label validation

PCA → UMAP on the HVGs, with **Harmony** batch correction over the 8 donors. The
donor UMAPs (before/after) show whether batch correction is warranted.

In [ ]:
preprocessing.embed(rna, hvg, run_harmony=True)
CT = rna.obs[LEVEL].astype(str).values
donor = rna.obs[config.DONOR_KEY].astype(str).values

In [ ]:
fig,_ = plotting.embedding_scatter(rna.obsm["X_umap"], CT); plotting.save(fig,f"fig_umap_celltype{TAG}"); fig

In [ ]:
fig,_ = plotting.embedding_scatter(rna.obsm["X_umap"], donor); plotting.save(fig,f"fig_umap_donor{TAG}"); fig

In [ ]:
fig,_ = plotting.embedding_scatter(rna.obsm["X_umap_harmony"], donor); plotting.save(fig,f"fig_umap_donor_harmony{TAG}"); fig

Cell-type labels validated against the **independent protein channel**: mean
CLR surface-protein level per cell type (z-scored per protein). The block-diagonal
structure (CD3→T, CD19/CD20→B, CD14→monocytes, CD56→NK, …) confirms the WNN labels
agree with direct molecular evidence.

In [ ]:
key = [p for p in ["CD3-1","CD4-1","CD8","CD19","CD20","CD14","CD16","CD56-1",
                    "CD11c","HLA-DR","CD123","CD34","CD27","CD25"] if p in adt.var_names]
clr = pd.DataFrame(np.asarray(adt.layers["clr"]), columns=adt.var_names); clr["ct"]=CT
mat = clr.groupby("ct")[key].mean(); matz = (mat-mat.mean())/mat.std()
fig,_ = plotting.protein_validation_heatmap(matz); plotting.save(fig,f"fig_adt_validation{TAG}"); fig

## 4. Protein-derived ground truth $D_c$

For each cell type we take the top surface proteins by one-vs-rest Wilcoxon
*score* on CLR-ADT (p-values are uninformative at n≈25k), mapreprocessinged to encoding
gene(s) by molecular fact only (`src.protein_gene_map`), intersected with the
scored gene universe. This is independent of all four RNA methods.

In [ ]:
drivers, details = ground_truth.build_ground_truth(adt, LEVEL, gene_universe=genes)
summary = ground_truth.summarise_drivers(drivers)
summary.to_csv(config.tab_path(f"ground_truth_summary{TAG}"), index=False)
summary

## 5. The four dependency measures

All operate on the shared matrix `X` and a binary one-vs-rest indicator.

### 5.1 Linear · marginal — Spearman correlation
### 5.2 Linear · conditional — partial correlation (shrinkage)
The empirical gene covariance is ill-conditioned (dropout, collinearity, $p\apreprocessingrox n$),
so a naïve precision matrix is unstable. We use a **Ledoit–Wolf shrinkage**
covariance → precision → point-biserial partial correlation. The eigen-spectrum
below motivates the shrinkage.

In [ ]:
from sklearn.covariance import LedoitWolf
Xs = X - X.mean(0, keepdims=True)
emp = np.linalg.eigvalsh(np.cov(Xs.T))
lw  = np.linalg.eigvalsh(LedoitWolf(assume_centered=True).fit(Xs).covariance_)
fig,_ = plotting.eigen_spectrum(emp, lw); plotting.save(fig,f"fig_pcorr_eigenspectrum{TAG}"); fig

### 5.3 Nonlinear · marginal — KSG mutual information
Kraskov–Stögbauer–Grassberger kNN estimator on the rank-transformed expression,
computed one-vs-rest per cell type (parallelised across cell types, estimated on a
fixed 10k-cell subsample for tractability).

In [ ]:
valid = [ct for ct,d in drivers.items()
         if len([g for g in d if g in set(genes)]) >= config.MIN_DRIVERS]
mi_attr = benchmark.compute_mi_parallel(X, rna.obs[LEVEL].astype(str).values, valid,
                                 n_jobs=-1, seed=config.SEED)
_ex = valid[0]
fig,_ = plotting.score_hist(mi_attr[_ex], f"mutual information — {_ex}"); plotting.save(fig,f"fig_mi_hist{TAG}"); fig

### 5.4 Nonlinear · conditional — Integrated Gradients on an MLP
A small classifier with `tanh` hidden layers and a `softmax` output (predictions
on the simplex), trained with cross-entropy + inverse-frequency class weights and
early stopreprocessinging on validation loss. Integrated Gradients then attributes each class
to the input genes.

**Architecture selection (CV macro-F1 sweep).** Overall accuracy hides
minority-class behaviour, so we select the `tanh`-MLP architecture by 3-fold
cross-validated **macro-F1** (every cell type weighted equally), which targets the
harder, less-abundant lineages. The winning configuration — hidden (512, 256),
dropout 0.4, lr 3e-4, weight decay 1e-4 — is the default in `src.mlp.train_mlp`.

In [ ]:
from src import mlp as mlpmod
y = rna.obs[LEVEL].astype(str).values
default_cfg = dict(hidden=(256,128), dropout=0.2, lr=1e-3, weight_decay=1e-5)
tuned_cfg   = dict(hidden=(512,256), dropout=0.4, lr=3e-4, weight_decay=1e-4)

At this granularity (30 classes) the sweep is expensive, so it is run
via `run_pipeline.py tune` and its results / per-class diagnostics are loaded here.

In [ ]:
from IPython.display import Image
sw = config.tab_path(f"mlp_hyperparam_search{TAG}")
display(pd.read_csv(sw)) if os.path.exists(sw) else None

In [ ]:
# per-class F1 (default vs tuned) and confusion matrix from the CV sweep
for _n in ("fig_mlp_per_class_f1", "fig_mlp_confusion"):
    _f = plotting.fig_dir("png") / f"{_n}.png"
    if _f.exists(): display(Image(str(_f)))

The final classifier is trained on all cells with the tuned architecture
(the `src.mlp` default), then Integrated Gradients is computed.

In [ ]:
trained = mlpmod.train_mlp(X, y, seed=config.SEED)
print(f"held-out test accuracy: {trained.test_acc:.3f}  (best epoch {trained.history.best_epoch})")
fig,_ = plotting.mlp_training_curves(trained.history); plotting.save(fig,f"fig_mlp_training{TAG}"); fig

In [ ]:
attr = mlpmod.integrated_gradients(trained, X, y, seed=config.SEED)
classes = list(trained.classes)
ig_attr = {ct: attr[:, classes.index(ct)] for ct in classes}
print("IG attributions:", attr.shape, "(genes × classes)")

UMAP of the MLP's penultimate-layer representation — the nonlinear,
class-discriminative geometry the attributions are read from.

In [ ]:
import umap
memb = mlpmod.mlp_embedding(trained, X)
mu = umap.UMAP(random_state=config.SEED).fit_transform(memb)
fig,_ = plotting.embedding_scatter(mu, CT); plotting.save(fig,f"fig_mlp_umap{TAG}"); fig

## 6. Benchmark: driver recovery ($\mathrm{AUC_{rel}}$)

$\mathrm{AUC_{rel}}$ is the normalized driver-recovery AUC (≡ Mann–Whitney ROC-AUC
of the per-gene score discriminating drivers from non-drivers): 0.5 = random,
1.0 = perfect, parameter-free, defined on every method's output.

In [ ]:
store = {}
res = benchmark.run_benchmark(X, genes, y, drivers, mi_attr=mi_attr, ig_attr=ig_attr,
                       store_scores=store, seed=config.SEED)
res.to_csv(config.tab_path(f"benchmark_auc_rel{TAG}"), index=False)
res.pivot_table(index="celltype", columns="method", values="auc_rel").round(3)

Cumulative recovery curve **averaged over cell types** (bootstrap-over
-cell-type band; x = fraction of all ranked genes). Its area is exactly
$\mathrm{AUC_{rel}}$.

In [ ]:
panel = {ct: {m: metric.recovery_curve(store[ct][m], store[ct]["_driver_mask"])
              for m in config.METHODS} for ct in sorted(store)}
fig,_ = plotting.recovery_curve_mean(store, config.METHODS); plotting.save(fig,f"fig_recovery_curves{TAG}"); fig

## 7. Statistical comparison

The unit of replication is the **cell type** (paired across methods): a Friedman
omnibus test, then Holm-corrected pairwise Wilcoxon signed-rank tests with
matched-pairs rank-biserial effect sizes.

In [ ]:
wide = stats.pivot_auc(res)
fried = stats.friedman_test(wide)
print(f"Friedman χ² = {fried['statistic']:.2f}, p = {fried['pvalue']:.2e} "
      f"(n = {fried['n_celltypes']} cell types)")
pw = stats.pairwise_wilcoxon(wide, method_order=config.METHODS)
pw.to_csv(config.tab_path(f"pairwise_wilcoxon{TAG}"), index=False)
pw

In [ ]:
sig = [(r.method_a, r.method_b, "*" if r["significant_0.05"] else "ns")
       for _, r in pw.iterrows()]
fig,_ = plotting.auc_box(res, config.METHODS, sig_pairs=sig); plotting.save(fig,f"fig_auc_box{TAG}"); fig

In [ ]:
fig,_ = plotting.auc_heatmap(wide, method_order=config.METHODS); plotting.save(fig,f"fig_auc_heatmap{TAG}"); fig

### 7.1 Early recovery (recall@k) and sensitivity to heterogeneous categories

$\mathrm{AUC_{rel}}$ integrates the *entire* ranking, but a practitioner only
inspects the top handful of genes. **Recall@k** (fraction of drivers within the
top-k) targets that early regime; the band is a bootstrap over cell types (the
replication unit). Note the axes differ from the per-cell-type curves above: here
x is the **absolute** top-k (a few dozen genes) **averaged over all cell types**,
whereas those curves use the *fraction* of all genes for a *single* type — so a
single-type curve reaching 100% early and a middling cross-type average are
consistent, not contradictory. The whole-curve winner and the early-recovery
winner need not agree.

**Recall@k averaged over cell types** (bootstrap-over-cell-type band);
x is the absolute top-k. The whole-curve winner and the early-recovery winner need
not agree, though here the marginal methods lead throughout.

In [ ]:
rk = benchmark.recovery_at_k_table(store, methods=config.METHODS)
rk.to_csv(config.tab_path(f"recovery_at_k{TAG}"), index=False)
fig,_ = plotting.recovery_at_k_curve(rk, config.METHODS); plotting.save(fig,f"fig_recovery_at_k{TAG}"); fig

Heterogeneous grab-bag categories (e.g. `other`, `other T`) have weak,
ambiguous protein-derived ground truth and depress recovery for *all* methods —
Integrated Gradients most. The table below (mean AUC_rel with vs without them) is a
transparency check showing how much the aggregate understates the model-based
method on cleanly-defined cell types.

In [ ]:
hetero = [c for c in res.celltype.unique() if "other" in c.lower()]
print("heterogeneous categories at this granularity:", hetero or "none")
sens = pd.DataFrame({"mean_auc_all": res.groupby("method").auc_rel.mean()})
if hetero:
    sens["mean_auc_excl_other"] = res[~res.celltype.isin(hetero)].groupby("method").auc_rel.mean()
sens.round(3).loc[config.METHODS]

## 8. Cross-method comparison (gene level)

Pooling every (gene, cell type) score (z-scored within cell type), how do the four
measures relate, and where do the protein-confirmed drivers sit? The pairwise
scatter matrix and the 3-D view (partial correlation × MI × IG) make the
agreement/disagreement structure explicit; protein-derived drivers are
highlighted.

In [ ]:
long = benchmark.scores_to_long(store, genes)
long.to_parquet(config.PROCESSED_DATA_DIR_PATH / f"scores_long{TAG}.parquet")
fig,_ = plotting.pairwise_scatter_matrix(long, config.METHODS); plotting.save(fig,f"fig_pairwise_scatter{TAG}"); fig

In [ ]:
fig,_ = plotting.scatter3d(long, "partial_corr", "mi_ksg", "ig_mlp"); plotting.save(fig,f"fig_scatter3d{TAG}", tight=False); fig

## 9. Stability and robustness

Descriptive stability bands (not inference): **bootstrap over cells** for the
methods, and **MLP-seed variation** for Integrated Gradients. At this granularity
the bootstrap recomputes MI / refits many shrinkage covariances, so it is produced
by `run_pipeline.py stability` / `seed` (CLI) and loaded here.

In [ ]:
sb, ss = config.tab_path(f"stability_bootstrap{TAG}"), config.tab_path(f"stability_seed{TAG}")
if os.path.exists(sb): display(pd.read_csv(sb, index_col=0).round(3))
if os.path.exists(ss): display(pd.read_csv(ss).round(3))

## 10. Summary

The figures and tables above decompose marker-gene identification along the
linear/nonlinear and marginal/conditional axes against an independent,
protein-derived ground truth, at this annotation granularity. Comparing against
the other granularity's notebook indicates whether the conclusion (which
statistical assumption changes the recovered markers) is robust to how finely cell
types are defined. See the report for interpretation.
